## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
groq_api_key= os.getenv("GROQ_API_KEY")
groq_api_key

'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'

In [9]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "Gemma2-9b-It", groq_api_key = groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001AC33F89B70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AC33F8AF20>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Yonas and I am AI Engineer Student")])

AIMessage(content="Hi Yonas, it's nice to meet you!  \n\nBeing an AI Engineer student is super exciting. What are you most interested in learning about or working on within AI?  \n\nI'm happy to chat about anything related to AI, or just have a general conversation. 😊  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 22, 'total_tokens': 86, 'completion_time': 0.116363636, 'prompt_time': 0.002129966, 'queue_time': 0.090376538, 'total_time': 0.118493602}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-fd200eba-8941-4c32-8bc8-898eefeff6fa-0', usage_metadata={'input_tokens': 22, 'output_tokens': 64, 'total_tokens': 86})

In [11]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Yonas and I am AI Engineer Student"),
        AIMessage(content="Hello Yonas! It's nice to meet you. \n\nAs a  AI Engineer Student, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="You're Yonas, and you're an AI Engineer Student!  \n\nIs there anything else you'd like to tell me about yourself or your work?  I'm happy to chat. 😊  \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 99, 'total_tokens': 146, 'completion_time': 0.085454545, 'prompt_time': 0.00434651, 'queue_time': 0.067121176, 'total_time': 0.089801055}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-1d92776c-1ca6-4558-a90c-576731083b61-0', usage_metadata={'input_tokens': 99, 'output_tokens': 47, 'total_tokens': 146})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!


In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory # for chain

store={}

# one user diffrent form other User or one chat distingush other chat 
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [17]:
config={"configurable":{"session_id":"chat1"}}

In [18]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")],
    config=config
)

In [19]:
response.content

"Hi Krish,\n\nIt's nice to meet you! That's an exciting title. What kind of AI projects are you working on these days?  \n\nI'm always eager to learn more about the cutting-edge work being done in the field of AI.\n"

In [20]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Krish.  \n\nYou told me at the beginning of our conversation! 😊  \n\nHow can I help you today, Krish?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 94, 'total_tokens': 126, 'completion_time': 0.058181818, 'prompt_time': 0.004276142, 'queue_time': 0.019655677, 'total_time': 0.06245796}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-6f7d0b7d-e991-4419-a8e0-2fe7721c6da4-0', usage_metadata={'input_tokens': 94, 'output_tokens': 32, 'total_tokens': 126})